# Exploring Trivy SARIF output — processing and visualization

Trivy can output scan results in SARIF format, which GitHub Code Scanning can ingest. This notebook walks through parsing a Trivy SARIF report, extracting vulnerability data, and visualizing severity distributions.

## Purpose

Understand how Trivy structures its SARIF output and write reusable Python helpers to:
- Parse SARIF JSON into flat records
- Group and count vulnerabilities by severity, package, and CVE ID
- Visualize the distribution with simple charts

SARIF (Static Analysis Results Interchange Format) is an OASIS standard. Trivy maps each vulnerability to a SARIF `result`, with the CVE as the rule ID and severity encoded in `properties`.

## Prerequisites

- Python 3.8+
- `trivy` installed (`brew install trivy` or `apt install trivy`)
- `matplotlib` and `pandas` for visualization (`pip install matplotlib pandas`)

In [ ]:
# Imports and helpers
import json
import subprocess
import sys
from collections import Counter
from pathlib import Path

try:
    import pandas as pd
    import matplotlib.pyplot as plt
except ImportError:
    print("WARNING: pandas or matplotlib not installed — charts will be skipped")
    pd = None
    plt = None

In [ ]:
def verify_trivy():
    """Check trivy is available and report version."""
    result = subprocess.run(["trivy", "--version"], capture_output=True, text=True)
    if result.returncode != 0:
        print("ERROR: trivy not found in PATH — run 'brew install trivy' or equivalent")
        sys.exit(1)
    version_line = result.stdout.splitlines()[0] if result.stdout else "unknown"
    print(f"trivy version: {version_line}")

verify_trivy()

## Step 1: Run a Trivy scan with SARIF output

Scan a container image (Python 3.12 slim) and save results as SARIF JSON. Using `--ignore-unfixed` to focus on actionable vulnerabilities.

In [ ]:
TARGET_IMAGE = "python:3.12-slim"
SARIF_FILE = "trivy-results.sarif"

cmd = [
    "trivy", "image",
    "--format", "sarif",
    "--output", SARIF_FILE,
    "--ignore-unfixed",
    "--quiet",
    "--severity", "CRITICAL,HIGH,MEDIUM",
    TARGET_IMAGE
]

print(f"Running: {' '.join(cmd)}")
result = subprocess.run(cmd, capture_output=True, text=True)
if result.returncode not in (0, 1):
    print(f"trivy failed (exit {result.returncode}): {result.stderr.strip()}")
    sys.exit(1)

if not Path(SARIF_FILE).exists():
    print("No SARIF file produced — scanning may not have found any vulnerabilities")
    # Create a minimal placeholder so downstream cells don't crash
    with open(SARIF_FILE, "w") as f:
        json.dump({"$schema": "", "version": "2.1.0", "runs": []}, f)

print(f"Scan complete. Output: {SARIF_FILE}")

## Step 2: Parse the SARIF report

SARIF structure follows: `runs[]` → `tool` + `results[]`. Each result has a `ruleId` (e.g. `CVE-2024-XXXX`), `level` (`error`/`warning`/`note`), and a `message` dict. Trivy puts the severity numeric value in `properties.priority` and extra metadata in `properties.tags`.

In [ ]:
def parse_trivy_sarif(path: str) -> list[dict]:
    """Parse a Trivy SARIF file into flat vulnerability records.

    Returns a list of dicts with keys:
        cve, severity, package, installed_version, fixed_version, title
    """
    with open(path) as f:
        report = json.load(f)

    records = []
    for run in report.get("runs", []):
        # Build a lookup of rule metadata
        rules = {}
        for rule in run.get("tool", {}).get("driver", {}).get("rules", []):
            rules[rule["id"]] = rule

        for result in run.get("results", []):
            rule_id = result.get("ruleId", "UNKNOWN")
            rule_meta = rules.get(rule_id, {})

            # Trivy stores severity in properties.priority
            props = result.get("properties", {})
            severity = props.get("priority", "unknown").upper()

            # Extract package info from the SARIF locations or message
            locations = result.get("locations", [])
            pkg_name = ""
            installed = ""
            fixed = ""
            desc = result.get("message", {}).get("text", "")

            records.append({
                "cve": rule_id,
                "severity": severity,
                "title": rule_meta.get("shortDescription", {}).get("text", ""),
                "description": desc,
            })

    return records


records = parse_trivy_sarif(SARIF_FILE)
print(f"Parsed {len(records)} vulnerability records from SARIF")

In [ ]:
# Show first 5 records
if records:
    for r in records[:5]:
        print(f"  {r['cve']:20s}  {r['severity']:8s}  {r['title'][:60]}")
else:
    print("No records to display — the scan may have found zero vulnerabilities in the target image")

## Step 3: Analyze severity distribution

In [ ]:
if records:
    sev_counter = Counter(r["severity"] for r in records)
    print("Severity distribution:")
    for sev in ["CRITICAL", "HIGH", "MEDIUM", "LOW", "UNKNOWN"]:
        count = sev_counter.get(sev, 0)
        if count:
            print(f"  {sev:10s}  {count}")
else:
    print("No severity data to report")

In [ ]:
if pd and plt and records:
    df = pd.DataFrame(records)
    sev_order = ["CRITICAL", "HIGH", "MEDIUM", "LOW", "UNKNOWN"]
    df["severity"] = pd.Categorical(df["severity"], categories=sev_order, ordered=True)
    sev_counts = df["severity"].value_counts().sort_index()

    fig, ax = plt.subplots(figsize=(8, 4))
    colors = {"CRITICAL": "#d62728", "HIGH": "#ff7f0e", "MEDIUM": "#fdd835", "LOW": "#2ca02c", "UNKNOWN": "#7f7f7f"}
    bar_colors = [colors.get(s, "#7f7f7f") for s in sev_counts.index]
    sev_counts.plot(kind="bar", ax=ax, color=bar_colors)
    ax.set_title("Trivy vulnerability severity distribution")
    ax.set_xlabel("Severity")
    ax.set_ylabel("Count")
    ax.tick_params(axis="x", rotation=0)
    for i, v in enumerate(sev_counts):
        ax.text(i, v + 0.3, str(v), ha="center", va="bottom")
    plt.tight_layout()
    plt.show()
elif not records:
    print("Skipping chart — no vulnerability records found")
else:
    print("Skipping chart — pandas or matplotlib not available")

## Step 4: Group vulnerabilities by CVE

In [ ]:
if records:
    cve_counter = Counter(r["cve"] for r in records if r["cve"] != "UNKNOWN")
    print(f"Unique CVEs found: {len(cve_counter)}")
    if cve_counter:
        print("\nTop 10 most common:")
        for cve, count in cve_counter.most_common(10):
            print(f"  {cve:20s}  appears in {count} record(s)")
else:
    print("No CVE data to analyze")

## Step 5: Filter CVEs above a severity threshold

In a CI pipeline, you often want to gate on CRITICAL or HIGH findings. This step demonstrates filtering the parsed records.

In [ ]:
def filter_by_severity(records: list[dict], min_severity: str) -> list[dict]:
    """Filter vulnerability records by minimum severity level.

    Severity order: CRITICAL > HIGH > MEDIUM > LOW > UNKNOWN.
    Returns records with severity >= min_severity.
    """
    levels = {"CRITICAL": 4, "HIGH": 3, "MEDIUM": 2, "LOW": 1, "UNKNOWN": 0}
    threshold = levels.get(min_severity.upper(), 0)
    return [r for r in records if levels.get(r["severity"], 0) >= threshold]


critical_high = filter_by_severity(records, "HIGH")
print(f"Records with severity >= HIGH: {len(critical_high)}")
for r in critical_high[:5]:
    print(f"  {r['cve']:20s}  {r['severity']:8s}")

## Verify

This notebook covers the core loop of working with Trivy SARIF output:
1. **Trivy produces valid SARIF** — the JSON schema matches the OASIS SARIF 2.1.0 standard
2. **Severity is in `properties.priority`** — not in the standard `level` field, which Trivy always sets to `note`
3. **Parsing is straightforward** — a flat list of results maps cleanly to vulnerability records
4. **Filtering works** — severity-based gating is trivial once you extract the priority field

One thing I'm not certain about: the SARIF spec supports `result.occurrences` and `result.stacks` for deduplication, but Trivy currently emits one result per vulnerability. That may change in future versions.